In [1]:
# Third-party core
import json
import boto3
import pandas as pd
from sqlalchemy import create_engine
import uuid
from pprint import pprint

# SageMaker
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.properties import PropertyFile
from sagemaker.pytorch.estimator import PyTorch
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.inputs import CreateModelInput
from sagemaker.workflow.model_step import ModelStep
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r
%store

Stored variables and their in-db values:
bucket                                     -> 'sagemaker-us-east-1-298748835671'
cls_model_data_url                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
cls_model_package_arn                      -> 'arn:aws:sagemaker:us-east-1:298748835671:model-pa
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                        -> True
landmarks_feature_group_name               -> 'landmarks-feature-group-08-23-58-13'
landmarks_table                            -> 'cat_annotations'
manifest_table                             -> 'image_manifest'
project_prefix                             -> 'cat-landmarks-project'
s3_athena_results_dir                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_cats_prefix                   -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_combined_prefix               

In [3]:
bucket = bucket
database_name = database_name
project_prefix = project_prefix
landmarks_table = landmarks_table
s3_staging_dir = s3_staging_dir
s3_raw_cats_prefix = s3_raw_cats_prefix
manifest_table = manifest_table

In [4]:
s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)


# SQL connection specification
engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

s3_athena_results_dir = f"s3://{bucket}/{project_prefix}/athena/results/"

In [5]:
# Retrieve feature group metadata
feature_groups = sagemaker_client.list_feature_groups()
feature_groups

{'FeatureGroupSummaries': [{'FeatureGroupName': 'landmarks-feature-group-17-18-53-51',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:298748835671:feature-group/landmarks-feature-group-17-18-53-51',
   'CreationTime': datetime.datetime(2026, 2, 17, 18, 53, 51, 418000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}},
  {'FeatureGroupName': 'landmarks-feature-group-08-23-58-13',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:298748835671:feature-group/landmarks-feature-group-08-23-58-13',
   'CreationTime': datetime.datetime(2026, 10, 8, 23, 58, 13, 516000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}}],
 'ResponseMetadata': {'RequestId': '510570d9-1fda-4653-a9f4-a8caf68d56da',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '510570d9-1fda-4653-a9f4-a8caf68d56da',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': '

In [6]:
# Instantiate the Landmarks Feature Group created in 4_Feature_store
fg_name = landmarks_feature_group_name

fg = FeatureGroup(
	name=fg_name,
	sagemaker_session=sagemaker_session
)

fg_refs = fg.describe()["OfflineStoreConfig"]["DataCatalogConfig"]
fg_refs

{'TableName': 'landmarks_feature_group_08_23_58_13_1791503893',
 'Catalog': 'AwsDataCatalog',
 'Database': 'sagemaker_featurestore'}

In [7]:
fg_training_data_query = f"""
	select 
 		remote_path,
   		split,
		width,
		height,
	 	left_eye_x,
	 	left_eye_y,
	    right_eye_x,
	    right_eye_y,
	    mouth_x,
	    mouth_y,
	    left_ear_1_x,
     	left_ear_1_y,
	    left_ear_2_x,
	    left_ear_2_y,
	    left_ear_3_x,
     	left_ear_3_y,
	    right_ear_1_x,
	    right_ear_1_y,
	    right_ear_2_x,
     	right_ear_2_y,
	    right_ear_3_x,
	    right_ear_3_y
 	from {fg_refs["Database"]}.{fg_refs["TableName"]}
"""

landmarks_sample = pd.read_sql(fg_training_data_query, engine)
landmarks_sample["split"].value_counts()

split
train         23991
production     7999
validation     1999
test           1999
Name: count, dtype: int64

In [8]:
training_uuid = uuid.uuid4()
training_uuid

UUID('bf21006a-1540-4c8a-ac03-503bad0b809a')

In [9]:
feature_store_data = f"s3://{bucket}/{project_prefix}/training-data/{training_uuid}"

fg_query = fg.athena_query()
fg_query.run(
    query_string=fg_training_data_query,
    output_location=feature_store_data
)
fg_query.wait()

INFO:sagemaker:Query b9b1d87f-7b37-4c22-8e8a-b5fd4debb71a is being executed.
INFO:sagemaker:Query b9b1d87f-7b37-4c22-8e8a-b5fd4debb71a successfully executed.


# Pipeline Parameters

In [10]:
# caching
cache_config = CacheConfig(enable_caching=True, expire_after="PT1H")

# Processing params
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount", default_value=1
    )

processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.g5.4xlarge"
    )

# Training Params
training_instance_count = ParameterInteger(
    name="TrainingInstanceCount", default_value=1
    )

training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.g5.4xlarge"
    )

# Evaluation Params
eval_instance_count = ParameterInteger(
    name="EvalInstanceCount", default_value=1
    )

eval_instance_type = ParameterString(
    name="EvalInstanceType", default_value="ml.g5.4xlarge"
    )

model_package_group_name = f"KeypointModelPackageGroupName"

model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="Approved"
)

mse_threshold = ParameterFloat(name="MseThreshold", default_value=0.10)

# Preprocessing Step

In [11]:
pipeline_session = PipelineSession()

image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.4xlarge",
    image_scope="training",
)

def make_processor():
    return ScriptProcessor(
        command=["python3"],
        image_uri=image_uri,
        role=role,
        instance_type="ml.g5.4xlarge",
        instance_count=1,
        sagemaker_session=pipeline_session,
    )

# Step process for training
step_process_train = ProcessingStep(
    name="KeypointPreprocessingTrain",
    cache_config=cache_config,
    processor=make_processor(),
    inputs=[
        ProcessingInput(
            source=feature_store_data,          
            destination="/opt/ml/processing/metadata"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",      
            source="/opt/ml/processing/output"
        )
    ],
    code="./src/keypoint_regression/preprocess.py",
    job_arguments=[
        "--metadata", "/opt/ml/processing/metadata",
        "--output",   "/opt/ml/processing/output",
        "--split",    "train",     
        "--sample-size", "5000"
    ],
)

# Step process for validation
step_process_val = ProcessingStep(
    name="KeypointPreprocessingValidation",
    cache_config=cache_config,
    processor=make_processor(),
    inputs=[
        ProcessingInput(
            source=feature_store_data,
            destination="/opt/ml/processing/metadata"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="validation",    
            source="/opt/ml/processing/output"
        )
    ],
    code="./src/keypoint_regression/preprocess.py",
    job_arguments=[
        "--metadata", "/opt/ml/processing/metadata",
        "--output",   "/opt/ml/processing/output",
        "--split",    "validation",
    ],
)

# Step process for testing
step_process_test = ProcessingStep(
    name="KeypointPreprocessingEval",
    cache_config=cache_config,
    processor=make_processor(),
    inputs=[
        ProcessingInput(
            source=feature_store_data,
            destination="/opt/ml/processing/metadata"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="test",    
            source="/opt/ml/processing/output"
        )
    ],
    code="./src/keypoint_regression/preprocess.py",
    job_arguments=[
        "--metadata", "/opt/ml/processing/metadata",
        "--output",   "/opt/ml/processing/output",
        "--split",    "test"
    ],
)

# Training Step

In [12]:
model_path = f"s3://{bucket}/{project_prefix}/models/model_artifacts"

pytorch_train = PyTorch(
    entry_point="train.py",
    source_dir="./src/keypoint_regression",
    role=role,
    framework_version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.4xlarge",
    instance_count=1,
    output_path=model_path,
    sagemaker_session=pipeline_session,
    hyperparameters={
        "epochs": 6,
        "learning-rate": 0.001,
        "batch-size": 64,
    },
)

train_args = pytorch_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process_train.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
        "validation": TrainingInput(
            s3_data=step_process_val.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
    }
)

step_train = TrainingStep(
    name="TrainKeypointModel",
    step_args=train_args,
    cache_config=cache_config
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


# Evaluation Step

In [13]:
script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type="ml.g5.4xlarge",
    instance_count=1,
    base_job_name="script-keypoint-reg-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    code="src/keypoint_regression/evaluation.py",
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process_test.properties
                .ProcessingOutputConfig
                .Outputs["test"]
                .S3Output
                .S3Uri,
            destination="/opt/ml/processing/test",
        ),
        # Training annotations only (not the images), for the mean-shape benchmark
        ProcessingInput(
            source=Join(on="/", values=[
                step_process_train.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
                "annotations.parquet",
            ]),
            destination="/opt/ml/processing/train",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        ),
    ], 
)

evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)

step_eval = ProcessingStep(
    name="KeypointModelEval",
    step_args=eval_args,
    property_files=[evaluation_report],
        job_arguments=[
        "--batch-size", "64"
    ],
)

# Create Model Step

In [14]:
# Serving image. train.py packages src/keypoint_regression/inference.py into model.tar.gz
# under code/, so the registered model deploys straight from the registry.
inference_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    image_scope="inference",
)

model = Model(
    image_uri=inference_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    env={
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    },
    sagemaker_session=pipeline_session,
    role=role,
)

step_create_model = ModelStep(
    name="KeypointCreateModel",
    step_args=model.create(instance_type="ml.g5.xlarge"),
)

# Batch Transform Step

In [15]:
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{project_prefix}/batch_transform/",
)

step_transform = TransformStep(
    name="KeypointTransform", transformer=transformer, inputs=TransformInput(data=None)
)

# Fail Step

In [16]:
step_fail = FailStep(
    name="KeypointMSEFail",
    error_message=Join(on=" ", values=["Execution failed due to MSE >", mse_threshold]),
)

# Registry Step

In [17]:
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["image/jpeg", "image/png", "application/x-npy"],
    response_types=["application/json"],
    inference_instances=["ml.g5.xlarge"],
    transform_instances=["ml.g5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)

step_register = ModelStep(name="KeypointRegisterModel", step_args=register_args)

# Condition Step

In [18]:
cond_lte = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.mse.value",
    ),
    right=mse_threshold,
)

step_cond = ConditionStep(
    name="KeypointMSECond",
    conditions=[cond_lte],
    if_steps=[step_register, step_create_model],
    else_steps=[step_fail],
)

# Pipeline Execution

In [19]:
pipeline = Pipeline(
    name="KeypointPipeline",
    parameters=[model_approval_status, mse_threshold],
    steps=[step_process_train, step_process_val, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

In [20]:
pipeline.upsert(role_arn=role)
execution = pipeline.start()
execution.wait()
print(execution.describe())

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 pipeline.upsert(role_arn=role)                                                               │
│   2 execution = pipeline.start()                                                                 │
│ ❱ 3 execution.wait()                                                                             │
│   4 print(execution.describe())                                                                  │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:938 in wait               │
│                                                                                                  │
│    935 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    936 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    937 │   │   )                                                                                 │
│ ❱  938 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    939 │                                                                                         │
│    940 │   def result(self, step_name: str):                                                     │
│    941 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:58 in wait                            │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:124 in wrapper                       │
│                                                                                                  │
│   121 │   │   │   with start_as_current_context():                                               │
│   122 │   │   │   │   if hook:                                                                   │
│   123 │   │   │   │   │   hook()                                                                 │
│ ❱ 124 │   │   │   │   return func(*args, **kwargs)                                               │
│   125 │   │                                                                                      │
│   126 │   │   return wrapper                                                                     │
│   127                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/waiter.py:

In [ ]:
for step in execution.list_steps():
    print(f"Step: {step['StepName']}")
    print(f"Status: {step['StepStatus']}")

    if step['StepStatus'] == 'Failed':
        print(f"Failure Reason: {step['FailureReason']}")
    print("---")

Step: KeypointCreateModel-CreateModel
Status: Succeeded
---
Step: KeypointRegisterModel-RegisterModel
Status: Succeeded
---
Step: KeypointMSECond
Status: Succeeded
---
Step: KeypointModelEval
Status: Succeeded
---
Step: TrainKeypointModel
Status: Succeeded
---
Step: KeypointPreprocessingEval
Status: Succeeded
---
Step: KeypointPreprocessingTrain
Status: Succeeded
---
Step: KeypointPreprocessingValidation
Status: Succeeded
---


In [ ]:
# Get the exact S3 path of the parquet file Athena wrote
for step in execution.list_steps():
    if step["StepStatus"] == "Failed":
        pprint(step)

# Pipeline Outputs

In [ ]:
# Resolve this execution's outputs for 6_Monitoring and 7_Endpoint_Demo
steps = {step["StepName"]: step for step in execution.list_steps()}

# Model artifact from the registered (condition-approved) model package
kp_model_package_arn = steps["KeypointRegisterModel-RegisterModel"]["Metadata"]["RegisterModel"]["Arn"]
kp_model_package = sagemaker_client.describe_model_package(ModelPackageName=kp_model_package_arn)
kp_model_data_url = kp_model_package["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]

# Pipeline processing outputs default to s3://<bucket>/<pipeline>/<execution id>/<step>/output/<output name>
execution_id = execution.arn.split("/")[-1]
kp_test_data_prefix = f"{pipeline.name}/{execution_id}/{step_process_test.name}/output/test"
s3.head_object(Bucket=bucket, Key=f"{kp_test_data_prefix}/annotations.parquet")

# Evaluation report attached to the registered model
evaluation_uri = kp_model_package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
evaluation_bucket, evaluation_key = evaluation_uri.replace("s3://", "").split("/", 1)
kp_evaluation = json.loads(s3.get_object(Bucket=evaluation_bucket, Key=evaluation_key)["Body"].read())

print("Model package:", kp_model_package_arn)
print("Model data:", kp_model_data_url)
print("Test data:", f"s3://{bucket}/{kp_test_data_prefix}")

%store kp_model_package_arn
%store kp_model_data_url
%store kp_test_data_prefix

# Test Set Evaluation

In [ ]:
# MSE/RMSE are in normalized coordinates (x / image width, y / image height).
# NME is each keypoint's pixel error divided by the inter-ocular distance (left_eye to right_eye);
# failure_rate_at_0_1 is the share of images with NME above 0.1.
print(f"Test samples: {kp_evaluation['test_samples']} (excluded, zero inter-ocular distance: {kp_evaluation['excluded_samples']})")

display(pd.DataFrame(
    {name: [metric["value"]] for name, metric in kp_evaluation["regression_metrics"].items()},
    index=["test"],
).round(4))

pd.Series(kp_evaluation["per_keypoint_nme"], name="mean NME").round(4).to_frame()

# Benchmark
The mean-shape benchmark predicts the average training keypoint positions for every image, ignoring the image itself. The model has to beat it to show that it locates each cat's face rather than reproducing the typical layout.

In [ ]:
mean_shape = kp_evaluation["benchmarks"]["mean_shape"]
benchmark_metrics = ["mse", "nme_mean", "nme_median", "nme_p95", "failure_rate_at_0_1"]

display(pd.DataFrame({
    "model": {name: kp_evaluation["regression_metrics"][name]["value"] for name in benchmark_metrics},
    "mean shape": {name: mean_shape[name] for name in benchmark_metrics},
}).round(4))

pd.DataFrame({
    "model": kp_evaluation["per_keypoint_nme"],
    "mean shape": mean_shape["per_keypoint_nme"],
}).round(4)